# Datavidia 11.0 — Task 2: Klasifikasi Fenomena dari Pasangan Citra

Notebook ini mereproduksi submission secara **inference-only**: bobot encoder
pure-vision publik (DINOv2 ViT-S/14 melalui timm), state normalisasi fitur, dan
probe klasifikasi dimuat dari folder `models/`. Tidak ada pelatihan ulang, tidak
ada label/ground truth, tidak ada prompt teks/VLM, dan tidak ada data eksternal
berlabel yang dipakai pada pelatihan.

Alur: manifest & thumbnail → statistik perubahan → embedding encoder beku →
fitur tersimpan → probe → Sinkhorn balance → `submissions/submission_main.csv`.

## Referensi Model Pretrained

**DINOv2 (vit_small_patch14_dinov2.lvd142m):**
- Paper: Oquab, M., et al. (2023). DINOv2: Learning Robust Visual Features without Supervision. arXiv:2304.07193. https://arxiv.org/abs/2304.07193
- Weights: https://huggingface.co/timm/vit_small_patch14_dinov2.lvd142m
- Original: https://github.com/facebookresearch/dinov2
- Method: Self-supervised learning on LVD-142M dataset, pure-vision (no text prompts)

**MobileNetV3 (mobilenetv3_large_100.ra_in1k):**
- Paper: Howard, A., et al. (2019). Searching for MobileNetV3. arXiv:1905.02244. https://arxiv.org/abs/1905.02244
- Weights: https://huggingface.co/timm/mobilenetv3_large_100.ra_in1k
- Pretrained: ImageNet-1k (supervised)

**timm library:**
- Wightman, R. (2019). PyTorch Image Models. https://github.com/huggingface/pytorch-image-models DOI: 10.5281/zenodo.4414861

Semua model menerima citra saja tanpa prompt teks, sesuai FAQ Task-2 no. 2.

In [1]:
import os, sys, csv
from pathlib import Path
import numpy as np
import torch

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src import config as C
from src.prepare import build_manifest, build_thumbs, build_change_stats
from src.features import build_features
from src.predict import probs_from_bundle, balanced_labels, write_submission, validate_submission

torch.set_num_threads(C.N_JOBS)
ENC = ["dinov2_s"]
SIZE = 196
STATE_TAG = "main"
PROBE_TAG = "main"
OUT_TAG = "final"
print('ROOT =', ROOT, '| encoders =', ENC, '| size =', SIZE)

ROOT = C:\Datavidia\TASK-2 | encoders = ['dinov2_s'] | size = 196


## 1. Data: hanya pasangan citra (tanpa label)

In [2]:
build_manifest()      # validasi 3.180 Id vs sample_submission.csv
build_thumbs()        # cache 256px agar inference lebih cepat
build_change_stats()  # 31 statistik perubahan piksel

[22:34:41] INFO prepare: manifest: 3180 pasangan -> C:\Datavidia\TASK-2\data\processed\manifest.csv
[22:34:43] INFO prepare: thumbnail siap: 0 berkas baru (total 6360)
[22:43:36] INFO prepare: change stats: 3180 baris x 33 fitur -> C:\Datavidia\TASK-2\data\processed\change_stats.csv


WindowsPath('C:/Datavidia/TASK-2/data/processed/change_stats.csv')

## 2. Inference encoder beku (pure-vision, tanpa prompt)

Bobot publik diunduh otomatis oleh `timm`; tidak ada proses training pada tahap ini.

In [3]:
from src.embed import run as run_embed
for enc in ENC:
    out = C.FEATURE_DIR / f'emb_{enc}_{SIZE}.npz'
    if not out.exists():
        run_embed(enc, SIZE)
    print('embedding:', out)

C:\Datavidia\TASK-2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


embedding: C:\Datavidia\TASK-2\cache\features\emb_dinov2_s_196.npz


## 3. Pemetaan klaster -> kelas (bukti visual, tanpa GT)

In [4]:
import json
mapping = json.loads((C.MODELS / 'cluster_mapping.json').read_text(encoding='utf-8'))
for c, name in sorted(mapping['cluster_to_class'].items()):
    print(f'klaster {c} -> {name} | {mapping.get("evidence", {}).get(c, "lihat cache/figures")}')

klaster 0 -> Earthquake | Perubahan tekstur ringan, sedikit gelap (gray_delta +0.008, edge_delta -0.007) — konsisten dengan keruntuhan tekstur tanpa pergeseran warna besar (bangunannya runtuh tapi lokasi sama).
klaster 1 -> Hurricane | Cahaya naik ringan (val_delta +0.02, sat_delta -0.01) — awan/hujan lebat meredam kontras, cocok dengan badai.
klaster 2 -> Tsunami | Perubahan paling ringan di semua klaster (diff_abs 0.182, val_delta ~0, sat_delta ~0) — air pasang tidak mengubah tekstur daratan banyak, cocok dengan tsunami.
klaster 3 -> Volcano | Perubahan terbesar (diff_abs 0.235), lebih gelap (dark_delta +0.057), lebih banyak tepi (edge_delta +0.010) — material vulkanik abu-abu/gelap dan lapisan baru, cocok dengan gunung meletus.
klaster 4 -> Explosion | Gelap drastis (dark_delta -0.124) dengan cahaya naik (val_delta +0.065) — ledakan membuat area hitam serta cahuan api, cocok dengan explosion.
klaster 5 -> Fire | Perubahan besar (diff_abs 0.238), gelap (dark_delta -0.038), tepi berku

## 4. Transform fitur + probe tersimpan (tanpa latih ulang)

In [5]:
import joblib
feat = build_features(ENC, state_tag=STATE_TAG)
bundle = joblib.load(C.MODELS / f'probe_{PROBE_TAG}.joblib')
probs = probs_from_bundle(bundle, feat['X'])
labels, balanced = balanced_labels(probs)
print('distribusi prediksi:', np.bincount(labels, minlength=6))

[22:43:52] INFO features: fitur: (3180, 417) dari blok ['post', 'stats'] (state=feature_state_main.joblib)
distribusi prediksi: [534 582 537 314 683 530]


## 5. Tulis & validasi submission

In [6]:
order = [r['Id'] for r in csv.DictReader(open(C.SAMPLE_SUBMISSION, encoding='utf-8'))]
pos = {pid: i for i, pid in enumerate(feat['ids'])}
ordered = np.array([labels[pos[p]] for p in order])
out = C.SUBMISSIONS / f'submission_{OUT_TAG}.csv'
write_submission(order, ordered, out)
print(out)
print(validate_submission(out))

C:\Datavidia\TASK-2\submissions\submission_final.csv
{'rows': 3180, 'class_counts': {'Earthquake': 534, 'Fire': 582, 'Hurricane': 537, 'Tsunami': 314, 'Volcano': 683, 'Explosion': 530}}
